In [ ]:
import os
import pandas as pd
import numpy as np
from glob import glob
from openslide import OpenSlide
import openslide as op
from pprint import pprint
from tqdm import tqdm 
import matplotlib.pyplot as plt
import seaborn as sns
import PIL.Image 


In [ ]:
def smart_sample(group, n=5, random_state=3):
    """Samples the minimum of n or the group size with no replacement."""
    n_samples = min(n, len(group))
    # replace=False ensures you get DISTINCT slide IDs
    return group.sample(n=n_samples, replace=False, random_state=random_state)
    
def openslide_script(counts_filename, original_filename, cluster_list): 
    #Replace this with the k-means pickle file format 
    counts_df = pd.read_csv(counts_filename)
    df = pd.read_csv(original_filename)
    df = pd.concat([df, counts_df['CHOIR_clusters_0.05']], axis=1) #need to fix this line, merging by ID but ID is slide not patch 
    df = df.rename(columns={'CHOIR_clusters_0.05': 'cluster'})
    
    for cluster in tqdm(cluster_list): 
        filtered_df = df[df['cluster'] == cluster].copy()
        unique_ids_df = filtered_df.drop_duplicates(subset=['ID', 'cancer_type'])
        #data = (unique_ids_df.groupby('cancer_type', group_keys=False).sample( n=5, replace=False, random_state=3).reset_index(drop=True)) 
        data = (unique_ids_df.groupby('cancer_type', group_keys=False).apply(smart_sample, n=5, random_state =3).reset_index(drop=True))
        FIXED_PLOT_SIZE = (256, 256) 
        j = 0
        plt.figure(figsize = (25, 25)) 
        for index, row in data.iterrows(): 
            j+=1
            cancer_type = row['cancer_type']
            slide_id = row['ID']
            x_coord = int(row['X'])
            y_coord = int(row['Y'])
            
            # 2. Open Slide and Calculate Read Size
            slide = op.OpenSlide('{}_new/{}.svs'.format(cancer_type.upper(), slide_id))
            
            # Get magnification, defaulting to 40 if property is missing/bad
            mag_prop = slide.properties.get('openslide.objective-power', '40')
            mag = int(mag_prop)
            
            # Your calculation for region size (num_read x num_read pixels at level 0)
            num_read = int(256 * mag / 40)
            
            # 3. Read Region and Resize
            img = slide.read_region(
                level = 0, 
                location = (x_coord, y_coord), 
                size = (num_read, num_read)
            )
            
            # === CRITICAL FIX: RESIZE IMAGE PATCH ===
            # Resize to the standard pixel size for consistent visual appearance
            img_resized = img.resize(FIXED_PLOT_SIZE, PIL.Image.Resampling.LANCZOS)
            
            # 4. Plotting
            # plt.subplot(nrows=5, ncols=5, index=j)
            plt.subplot(5, 5, j)
            plt.imshow(img_resized)
            
            # Set the title
            plt.title('{},{},{}'.format(slide_id[:12], cancer_type, mag), fontsize=10)
            plt.axis('off')
            
            # Close the slide object when done with it
            slide.close() 
            plt.savefig('250k_sample/cluster_{}.png'.format(cluster))


In [ ]:
def smart_sample(group, n=5, random_state=3):
    """Samples the minimum of n or the group size with no replacement."""
    n_samples = min(n, len(group))
    # replace=False ensures you get DISTINCT slide IDs
    return group.sample(n=n_samples, replace=False, random_state=random_state)
    
def openslide_script(original_filename, cluster_list): 
    df = pd.read_pickle(original_filename)
    df = df.reset_index()
    df.columns = ['cancer_type', 'ID', 'seq_num', 'label', 'X', 'Y']   
    target_cancers = ['coad', 'esca', 'paad', 'stad', 'read']
    new_df = df[df['cancer_type'].isin(target_cancers)]
    new_df = new_df[['cancer_type', 'ID', 'label', 'X', 'Y']]
    
    for cluster in tqdm(cluster_list): 
        filtered_df = new_df[new_df['label'] == cluster].copy()
        unique_ids_df = filtered_df.drop_duplicates(subset=['ID', 'cancer_type'])
        #data = (unique_ids_df.groupby('cancer_type', group_keys=False).sample( n=5, replace=False, random_state=3).reset_index(drop=True)) 
        data = (unique_ids_df.groupby('cancer_type', group_keys=False).apply(smart_sample, n=5, random_state =3).reset_index(drop=True))
        FIXED_PLOT_SIZE = (256, 256) 
        j = 0
        plt.figure(figsize = (25, 25)) 
        for index, row in data.iterrows(): 
            j+=1
            
            cancer_type = row['cancer_type']
            slide_id = row['ID']
            x_coord = int(row['X'])
            y_coord = int(row['Y'])
            
            # 2. Open Slide and Calculate Read Size
            slide = op.OpenSlide('{}_new/{}.svs'.format(cancer_type.upper(), slide_id))
            
            # Get magnification, defaulting to 40 if property is missing/bad
            mag_prop = slide.properties.get('openslide.objective-power', '40')
            mag = int(mag_prop)
            
            # Your calculation for region size (num_read x num_read pixels at level 0)
            num_read = int(256 * mag / 40)
            
            # 3. Read Region and Resize
            img = slide.read_region(
                level = 0, 
                location = (x_coord, y_coord), 
                size = (num_read, num_read)
            )
            
            # === CRITICAL FIX: RESIZE IMAGE PATCH ===
            # Resize to the standard pixel size for consistent visual appearance
            img_resized = img.resize(FIXED_PLOT_SIZE, PIL.Image.Resampling.LANCZOS)
            
            # 4. Plotting
            # plt.subplot(nrows=5, ncols=5, index=j)
            plt.subplot(5, 5, j)
            
            plt.imshow(img_resized)
            
            # Set the title
            plt.title('{},{},{}'.format(slide_id[:12], cancer_type, mag), fontsize=10)
            plt.axis('off')
            
            # Close the slide object when done with it
            slide.close() 
            plt.savefig('cluster_60/cluster_{}.png'.format(cluster))

          

In [ ]:
counts_df = pd.read_csv('csv/250k_sample_counts.csv')
counts_df.head()

In [ ]:
df = pd.read_csv('csv/250k_sample.csv')
df.head()

In [ ]:
new_df = pd.read_pickle('GI_cluster60_labelled_all_02_18_26.pkl')
new_df.head()

In [ ]:
# 1. Reset the index to turn index levels into columns
new_df = new_df.reset_index()

# 2. Rename the columns (adjust 'level_0' etc. based on your specific index names)
# Based on the image, level_0 is 'cancer_type' and level_1 is the 'slide_id'
new_df.columns = ['cancer_type', 'full_id', 'seq_num', 'label', 'X', 'Y']

# 3. Extract the short ID (the first 12 characters, e.g., 'TCGA-HZ-A9TJ')
new_df['ID'] = new_df['full_id'].str[:12]

# 4. Optional: Drop the old long ID and sequence column if they are no longer needed
new_df = new_df[['cancer_type', 'ID', 'label', 'X', 'Y']]

new_df.head()

In [ ]:
cluster_list = [i for i in range(60)]
openslide_script('GI_cluster60_labelled_all_02_18_26.pkl', cluster_list)

In [ ]:
cluster_list = [i for i in range(40)]
openslide_script('GI_cluster40_labelled_all_02_18_26.pkl', cluster_list)
#Need to rerun 40 because I used the cluster 60 data initially 